# Single-Field Cube Imaging with AstroVIPER

[Colab Link](https://colab.research.google.com/github/casangi/astroviper/blob/main/docs/distributed_applications_tutorials/imaging/single_field_cube.ipynb)


**AstroVIPER** (Astro **V**isibility and **I**mage **P**arallel **E**xecution **R**eduction)
is the *science* package of the VIPER radio-astronomy ecosystem:

| Package | Role |
| --- | --- |
| **[ToolVIPER](https://github.com/casangi/toolviper)** | Logging, parameter checking, and Dask cluster tools |
| **[XRADIO](https://xradio.readthedocs.io/en/latest/)** | I/O + data model: Processing Sets, Measurement Set v4, image datasets |
| **[GraphVIPER](https://graphviper.readthedocs.io/en/latest/)** | Concurrency: builds and runs Dask MapReduce graphs |
| **AstroVIPER** | Science: gridding, weighting, FFT, deconvolution (CLEAN), imaging workflows |
| **[FlowVIPER](https://github.com/casangi/flowviper)** | Data-processing workflows built from AstroVIPER's distributed applications |

This tutorial uses the **single-field cube imager** — `image_cube_single_field` — as a worked
example of how to run AstroVIPER:

1. Run the imager end-to-end on ALMA TW Hydrae data (dirty image, then a multi-cycle CLEAN).
2. Tour the inputs and outputs: Processing Sets, image datasets, and **data groups**.
3. See how the imager maps onto AstroVIPER's four-layer architecture.

For the developer-facing details — layer rules, coding conventions, and the imaging
deep-dive — see [`AGENTS.md`](../../../AGENTS.md) at the repository root.

**Prerequisites**: `pip install astroviper` (builds the C++ extensions, so it needs a compiler;
see the repository README for developer installs). The dataset used here is the same
5-channel TW Hydrae processing set used by the component tests in
`tests/component/test_single_field_imaging.py`.

## 1. Start a Dask client

GraphVIPER executes the imaging graph on whatever Dask scheduler is active. For interactive
work, start a local cluster with toolviper's `local_client`; the dashboard link it prints is
handy for watching the frequency chunks execute in parallel. (For strictly deterministic
debugging you can instead run everything in the current process with
`dask.config.set(scheduler="synchronous")`.)

In [ ]:
from toolviper.dask.client import local_client

viper_client = local_client(cores=4, memory_limit="4GB")
viper_client

## 2. Get the data

**This is temporary, we will move this to Cloudflare.**
The input is a **Processing Set**: a Zarr store holding one or more Measurement Set v4s.
This one contains TW Hydrae ALMA data, split to 5 LSRK channels. It is distributed as a
zipped Zarr directory on Google Drive (the download is skipped when the directory is
already present).

In [ ]:
import glob
import os
import shutil
import zipfile

PS_STORE = "twhya_selfcal_lsrk_5chans.ps.zarr"
PS_STORE_DRIVE_ID = "1BRe3cD6YAWkn-jSPbClGGM9VlbxHP_yn"


def download_zarr_from_drive(zarr_name, file_id):
    """Download and extract one zipped ``.zarr`` directory from Google Drive.

    A no-op when the directory already exists locally. Handles archives that
    arrive double-zipped (a ``.zip`` whose only member is another ``.zip``).
    """
    if os.path.isdir(zarr_name):
        return  # already present locally -- nothing to download.

    import gdown

    zip_path = zarr_name + ".zip"
    gdown.download(id=file_id, output=zip_path, quiet=False)

    work_dir = zarr_name + ".extract"
    shutil.rmtree(work_dir, ignore_errors=True)
    os.makedirs(work_dir)
    shutil.move(zip_path, os.path.join(work_dir, os.path.basename(zip_path)))

    for _ in range(6):  # safety bound against malformed archives
        for root, dirs, _files in os.walk(work_dir):
            if zarr_name in dirs:
                shutil.move(os.path.join(root, zarr_name), zarr_name)
                shutil.rmtree(work_dir, ignore_errors=True)
                return
        nested_zips = glob.glob(os.path.join(work_dir, "**", "*.zip"), recursive=True)
        if not nested_zips:
            break
        for nested in nested_zips:
            with zipfile.ZipFile(nested) as zf:
                zf.extractall(os.path.dirname(nested))
            os.remove(nested)

    shutil.rmtree(work_dir, ignore_errors=True)
    raise RuntimeError(f"Could not extract '{zarr_name}' from its archive.")


download_zarr_from_drive(PS_STORE, PS_STORE_DRIVE_ID)

## 3. Explore the Processing Set

A Processing Set (`ps_xdt`) is an `xarray.DataTree` whose children are individual
**Measurement Set v4s** — self-describing correlated data for one observation / spectral
window / polarization setup, with dimensions `time x baseline_id x frequency x polarization`.

Two naming conventions from XRADIO to remember:

- `open_*` functions are **lazy** (metadata only; data variables are lazily loaded), while
  `load_*` functions are **eager** (everything into memory now). The driver opens the
  processing set lazily and each parallel task loads only its own chunk.
- **Coordinates** are lowercase (`frequency`, `time`, `polarization`); **data variables** are
  UPPERCASE (`VISIBILITY`, `WEIGHT`, `FLAG`, `UVW`).

The `xr_ps` accessor provides processing-set-level helpers such as `summary()`.

In [ ]:
from xradio.measurement_set import open_processing_set

ps_xdt = open_processing_set(PS_STORE)
ps_xdt.xr_ps.summary()

### Data groups

A **data group** lets one dataset hold multiple versions of the same logical variable
(raw vs. corrected vs. model visibilities, say) without overwriting anything. It is a dict
stored in `attrs["data_groups"]` mapping a group name to *logical roles* (lowercase keys:
`correlated_data`, `flag`, `weight`, `uvw`) whose values are *data-variable names*
(UPPERCASE). Groups may share variables. Below, the imager is pointed at a group with
`processing_set_data_group_name`; everything it reads (visibilities, flags, weights, uvw)
is resolved through that group.

In [ ]:
ms_name, ms_xdt = list(ps_xdt.items())[0]
print(f"Measurement set: {ms_name}")
ms_xdt.attrs["data_groups"]

## 4. Imaging parameters

The driver takes three parameter dicts, validated once at the graph layer against a JSON
schema (`image_cube_single_field.param.json`) that lives next to the driver module — see
"Coding Conventions" in `AGENTS.md`:

**`image_params`** — the image geometry and output coordinates:

- `image_size` — pixels along `l` and `m`.
- `cell_size` — pixel size in **radians**; the first (l / right-ascension) element is negative
  by convention so RA increases leftward.
- `phase_direction` — image phase center; taken from the processing set's combined
  field-and-source dataset below.
- `frequency_coords` — output cube channels; here simply the processing set's frequency axis.
- `polarization_coords` — the **Stokes** basis of the output image (gridding itself happens in
  the instrument's correlation basis — `XX`/`YY` for ALMA's linear feeds — controlled by the
  separate `instrument_polarization_basis` argument).
- `time_coords` — output time axis (`[0]` collapses all times into one plane).
- `fft_padding` — padding factor applied to the uv grid before the FFT.

**`imaging_weights_params`** — `"natural"` or `"briggs"` weighting (with the usual `robust`
parameter).

**`iteration_control_params`** — the CLEAN controls, matching CASA `tclean` semantics with one
deliberate difference: iteration control is **independent per (time, frequency, polarization)
plane**. Each plane has its own `niter` budget and `threshold`, and residual update cycles
continue until *every* plane has stopped. The main knobs: `niter` (max CLEAN components per
plane), `nmajor` (max residual update cycles; `-1` = unlimited), `threshold` (Jy, absolute
stop), `gain` (loop gain), `primary_beam_limit` (mask cutoff as a fraction of the peak primary
beam), and `cyclefactor` / `cycleniter` / `minpsffraction` / `maxpsffraction` (when a model
update cycle hands control back to the next residual update cycle). We will change this in the future to be more descriptive.

In [ ]:
import numpy as np

combined_field_xds = ps_xdt.xr_ps.get_combined_field_and_source_xds()
center_field_name = combined_field_xds.attrs["center_field_name"]
phase_direction = combined_field_xds.FIELD_PHASE_CENTER_DIRECTION.sel(
    field_name=center_field_name
)

image_params = {
    "image_size": [250, 250],
    "cell_size": np.array([-0.1, 0.1]) * np.pi / (180 * 3600),  # 0.1 arcsec pixels
    "phase_direction": phase_direction.values,
    "frequency_coords": ps_xdt.xr_ps.get_freq_axis().values,
    "polarization_coords": ["I", "Q"],
    "time_coords": [0],
    "fft_padding": 1.2,
    "cpp_gridder": True,
}

imaging_weights_params = {
    "weighting": "briggs",
    "robust": 0.5,
    "casa_weighting_implementation": True,
}

## 5. Make a dirty image (`niter=0`)

With `niter=0` the imager performs no deconvolution: it computes imaging weights, grids the
visibilities, and FFTs them into the **dirty image** (stored as `SKY_RESIDUAL` — with no model
subtracted, the residual *is* the dirty image), along with the **point spread function** and
the **primary beam**.

`image_data_variables_keep` selects which image variables are written to the output Zarr
store; everything else is discarded when the per-chunk task finishes. `n_mapping_parallelism` sets the mapping parallelism as
`{parallel_axis: n_chunks}` — for cube imaging always `{"frequency": ...}`: how many
frequency chunks the cube is split into (one Dask task each;
`None` auto-picks from a per-channel memory estimate), while `processing_function_threads` is
the *within*-task thread count handed to the C++/FFT kernels — the two levels of
parallelism multiply, so keep `frequency chunks x processing_function_threads` near your core count.

In [ ]:
from astroviper.distributed_applications.imaging import image_cube_single_field

DIRTY_IMAGE_STORE = "twhya_dirty.img.zarr"

dirty_iteration_control = {
    "niter": 0,
    "nmajor": 0,
    "threshold": 0.0,
    "gain": 0.1,
    "cyclefactor": 1.5,
    "cycleniter": -1,
    "minpsffraction": 0.05,
    "maxpsffraction": 0.8,
}

return_dict_dirty = image_cube_single_field(
    ps_store=PS_STORE,
    image_store=DIRTY_IMAGE_STORE,
    image_params=image_params,
    imaging_weights_params=imaging_weights_params,
    iteration_control_params=dirty_iteration_control,
    gridder="prolate_spheroidal",
    deconvolver="hogbom_many_threads",
    scan_intents="OBSERVE_TARGET#ON_SOURCE",
    image_data_variables_keep=[
        "sky_residual",
        "point_spread_function",
        "primary_beam",
        "beam_fit_params_point_spread_function",
    ],
    processing_set_data_group_name="base",
    single_precision_image=False,
    processing_function_threads=1,
    n_mapping_parallelism={"frequency": 5},
    overwrite=True,
)

The returned dict contains four things: the merged per-plane deconvolution statistics
(`"deconvolution"`, empty here since `niter=0`), per-plane **image statistics** of every image
variable that was in memory in the node tasks (`"image_statistics"`, shown in section 6), a
`pandas.DataFrame` of per-chunk node-task timings (`"timing_node_tasks"`), and the driver-level
step timings (`"timing_distributed_application"`).

In [ ]:
return_dict_dirty["timing_node_tasks"]

### Driver-level timing breakdown

`return_dict["timing_distributed_application"]` holds the per-step timings of the
**driver** (the distributed application itself): building and writing the empty
image, constructing the map/reduce graph, generating and computing the Dask
graph, and consolidating the Zarr metadata -- plus the grand total. AstroVIPER
ships the phase layout and a formatter, so the breakdown prints the same way the
imager logs it.

In [ ]:
from astroviper.distributed_applications.imaging.image_cube_single_field import (
    DISTRIBUTED_APPLICATION_TIMING_PHASES,
    DISTRIBUTED_APPLICATION_TIMING_TOTAL_KEY,
)
from astroviper.utils.timing import format_timing_summary

print(
    format_timing_summary(
        return_dict_dirty["timing_distributed_application"],
        DISTRIBUTED_APPLICATION_TIMING_PHASES,
        total_key=DISTRIBUTED_APPLICATION_TIMING_TOTAL_KEY,
        title="AstroVIPER distributed-application timing (driver, seconds)",
        total_label="TOTAL (driver wall time)",
    )
)

### The output image dataset

The image store is a Zarr dataset with two coordinate spaces: **image domain** dims
`(time, frequency, polarization, l, m)` and (when kept) **uv/grid domain** dims
`(time, frequency, polarization, u, v)`. Image datasets carry **data groups** too —
image-side roles like `sky`, `point_spread_function`, `primary_beam`, and `mask` map to
the UPPERCASE variables on disk (see "Data Groups" in `AGENTS.md`). The driver registers
one group per product for the variables in `image_data_variables_keep`: here a
`residual` group (the dirty image and its PSF/primary-beam companions); a CLEAN run adds
`model` and `restored` groups.

In [ ]:
import xarray as xr

dirty_xds = xr.open_zarr(DIRTY_IMAGE_STORE)
print("Image data groups:")
for name, group in dirty_xds.attrs["data_groups"].items():
    print(f"  {name}: {group}")
dirty_xds

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

chan = 2  # middle channel
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), constrained_layout=True)
for ax, var in zip(
    axes, ["SKY_RESIDUAL", "POINT_SPREAD_FUNCTION", "PRIMARY_BEAM"], strict=False
):
    plane = dirty_xds[var].isel(time=0, frequency=chan, polarization=0)
    im = ax.imshow(plane.values, origin="lower", cmap="viridis")
    ax.set_title(f"{var}\nchannel {chan}, Stokes I")
    fig.colorbar(im, ax=ax)
plt.show()

## 6. CLEAN: residual update and model update cycles

Now a real deconvolution. Each **residual update cycle** (CASA's *major cycle*) degrids the
current model into model visibilities, subtracts them from the observed visibilities *in
double precision*, and grids and FFTs the difference into a residual image. Each **model
update cycle** (CASA's *minor cycle* — the deconvolver, here Hogbom CLEAN) then picks peaks
in the residual, inside a mask built from the primary-beam limit, and moves flux into the
model image until `cycleniter` / `cyclethreshold` triggers the next residual update cycle. A
final residual update cycle always follows the last model update cycle so the reported
residual is consistent with the final model.

With `restore=True` the model is convolved with the **clean beam** (the per-channel Gaussian
fit to the PSF, stored in `BEAM_FIT_PARAMS_POINT_SPREAD_FUNCTION`) and added to the residual,
producing `SKY_RESTORED`.

Two things worth noting in the call below:

- `single_precision_image=True` (the default) keeps the *image-domain* arrays — uv grids and
  sky/PSF/model images — in single precision, halving the cube's memory footprint. The
  visibilities always stay `complex128`, and the residual is always formed in double
  precision.
- `sky_model` and `mask` are added to the keep list so the CLEAN products are written out.

In [ ]:
CLEAN_IMAGE_STORE = "twhya_clean.img.zarr"

clean_iteration_control = {
    "niter": 300,
    "nmajor": 3,
    "threshold": 0.001,  # Jy
    "primary_beam_limit": 0.2,
    "gain": 0.1,
    "cyclefactor": 1.5,
    "cycleniter": -1,
    "minpsffraction": 0.05,
    "maxpsffraction": 0.8,
}

return_dict_clean = image_cube_single_field(
    ps_store=PS_STORE,
    image_store=CLEAN_IMAGE_STORE,
    image_params=image_params,
    imaging_weights_params=imaging_weights_params,
    iteration_control_params=clean_iteration_control,
    gridder="prolate_spheroidal",
    deconvolver="hogbom_many_threads",
    scan_intents="OBSERVE_TARGET#ON_SOURCE",
    image_data_variables_keep=[
        "sky_residual",
        "sky_model",
        "mask",
        "point_spread_function",
        "primary_beam",
        "beam_fit_params_point_spread_function",
    ],
    processing_set_data_group_name="base",
    single_precision_image=True,
    processing_function_threads=1,
    n_mapping_parallelism={"frequency": 5},
    overwrite=True,
    restore=True,
    # Sample each node task's CPU / memory / I/O every 0.05 s; the series are
    # returned as list-valued columns of "timing_node_tasks" (plotted below).
    monitor_resources_seconds=0.05,
)

`return_dict["deconvolution"]` is a `ReturnDict` of per-plane convergence statistics keyed by
`(time, polarization, channel)`: iterations done, peak residual per cycle, the mask size, and
a CASA-style stop code explaining *why* each plane stopped.

In [ ]:
from astroviper.processing_functions.imaging.utils import print_deconvolve_dict

print_deconvolve_dict(return_dict_clean["deconvolution"])

### Per-plane image statistics

Right before each node task writes its chunk, it computes NaN-ignoring statistics of every
image-domain variable it holds (`sky_residual`, `sky_model`, `sky_restored`, ...) over the
`(l, m)` axes of each `(time, frequency, polarization)` plane; the reduce concatenates the
chunks along `frequency`. `return_dict["image_statistics"]` is a dict of
`xarray.Dataset`s with dims `(time, frequency, polarization)` and one variable per statistic:
`mean`, `median`, `max`, `min`, `peak` (signed value of the pixel with the largest |value| --
the CLEAN peak residual), `sum`, `rms`, `std`, `mad_sigma` (1.4826 x MAD, a noise estimate
robust to sources) and `n_pixels`, each with a `_masked` twin over the valid (not-masked-out) area: the clean mask when the run
deconvolved, else the fallback `PRIMARY_BEAM > primary_beam_limit` (so `niter=0` runs get
masked statistics too; the `mask_source` attr records which was used). They describe exactly
what went to disk, with no need to re-read the cube.

In [ ]:
image_statistics = return_dict_clean["image_statistics"]
print("Variables with statistics:", list(image_statistics))
image_statistics["sky_residual"]

In [ ]:
stat_panels = [
    ("peak", "Signed peak"),
    ("rms", "RMS"),
    ("mad_sigma", "MAD sigma (robust noise)"),
    ("mean", "Mean"),
    ("median", "Median"),
    ("sum", "Sum over pixels"),
]
variables = [
    v for v in ("sky_residual", "sky_model", "sky_restored") if v in image_statistics
]
fig, axes = plt.subplots(
    len(stat_panels),
    len(variables),
    figsize=(5 * len(variables), 2.6 * len(stat_panels)),
    squeeze=False,
    constrained_layout=True,
)
for col, variable in enumerate(variables):
    stats = image_statistics[variable].isel(time=0)
    channel = np.arange(
        stats.sizes["frequency"]
    )  # frequency values: stats["frequency"]
    for row, (stat, title) in enumerate(stat_panels):
        ax = axes[row, col]
        for i, pol in enumerate(stats["polarization"].values):
            ax.plot(
                channel,
                stats[stat].sel(polarization=pol),
                "o-",
                color=f"C{i}",
                label=str(pol),
            )
            ax.plot(
                channel,
                stats[stat + "_masked"].sel(polarization=pol),
                "s--",
                color=f"C{i}",
                label=f"{pol} (masked)",
            )
        ax.set_title(f"{variable}: {title}", fontsize=10)
        ax.set_ylabel("Jy/beam")
        ax.grid(True, color="lightgray")
    axes[-1, col].set_xlabel("Channel")
axes[0, 0].legend(fontsize=8)
plt.show()

Solid lines are over all pixels, dashed over the clean-mask pixels only. Compare the residual's
`peak_masked` per channel with the final peak residual in the deconvolution dict above: they
are the same quantity, measured on the cube that was written. `rms` versus `mad_sigma` of the
residual tells you how much remaining structure there is above the noise, and the `sum` of the
model is the total cleaned flux per channel.

### Per-task resource usage (CPU / memory / I/O)

Because the CLEAN run was launched with `monitor_resources_seconds=0.25`, a
sampler thread recorded each node task's worker-process resource usage while it
ran (this needs `psutil`; per-task attribution is exact with one concurrent
task per worker process). The series arrive as list-valued columns of
`timing_node_tasks`, and `astroviper.utils.resource_plots` renders the two
standard views of them.

**Task-aligned view** -- every task's clock starts at its own beginning; the
bold curve is the average over the tasks *still running* at each instant (so it
is not diluted by tasks that already finished). `cpu_percent` covers all
threads of the worker process (OpenMP included), so >100% is normal, and the
cumulative I/O counters are rebased per task.

In [ ]:
from astroviper.utils.resource_plots import (
    plot_cluster_resource_usage,
    plot_task_resource_usage,
    plot_task_stream,
)

plot_task_resource_usage(return_dict_clean);

**Cluster view over the entire run** -- each task is placed at its actual
wall-clock position (the monitor records a `start_unixtime` anchor per task)
and the quantities are **summed over the tasks running at each moment**: busy
cores, total resident memory of task-running workers, and the aggregate
read/write rate. The gray curve (right axis) is the number of concurrently
running tasks, so ramp-up, steady state, and the straggler tail are visible at
a glance.

In [ ]:
plot_cluster_resource_usage(return_dict_clean);

**Task stream** -- a Dask-dashboard-style per-worker timeline: one lane
per worker process, each task drawn as load (blue) / science (green) /
write (orange) segments, with a running-task utilization panel on top and
a printed efficiency decomposition (ideal time, ramp-up, straggler tail,
pre/post-map regions). Reduce nodes are drawn (pink) on the worker lane
they actually ran on. Unlike the two views above, this needs only the
per-task `start_unixtime` anchor the node task records itself, **not** the
monitor's sampled series -- so it also works for runs launched with
`monitor_resources_seconds=None`.


In [ ]:
plot_task_stream(return_dict_clean);

In [ ]:
clean_xds = xr.open_zarr(CLEAN_IMAGE_STORE)
print("Image data groups:")
for name, group in clean_xds.attrs["data_groups"].items():
    print(f"  {name}: {group}")

chan = 2
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), constrained_layout=True)
for ax, var in zip(axes, ["SKY_RESIDUAL", "SKY_MODEL", "SKY_RESTORED"], strict=False):
    plane = clean_xds[var].isel(time=0, frequency=chan, polarization=0)
    im = ax.imshow(plane.values, origin="lower", cmap="viridis")
    ax.set_title(f"{var}\nchannel {chan}, Stokes I")
    fig.colorbar(im, ax=ax)
plt.show()

The clean beam per channel (from the Gaussian PSF fit, in radians:
major FWHM, minor FWHM, position angle):

In [ ]:
clean_xds["BEAM_FIT_PARAMS_POINT_SPREAD_FUNCTION"].isel(
    time=0, polarization=0
).to_pandas()

## 7. Writing the image as FITS files (`output_image_format="fits"`)

The imager can write its output as **FITS files** instead of a Zarr store, using the same
parallel-write pattern the Zarr path uses:

1. The **driver** pre-creates one XRADIO-conformant FITS file per kept image variable —
   `<image_store>/<VARIABLE>.fits` — with the complete header (direction/Stokes/frequency
   axes, reference frame, rest frequency, observation date, telescope) and a **sparse**
   data area sized for the full cube, so creation costs only metadata.
2. Each **node task** then `pwrite`s its frequency chunk directly into the shared files.
   Frequency is deliberately the *slowest-varying* FITS axis, so a task's block of
   consecutive channels is one **contiguous byte range**: parallel tasks write disjoint
   ranges with no locking and no file creation (the same "single parallel file" pattern as
   the sharded Zarr writer).

The per-channel clean-beam fits (`beam_fit_params_point_spread_function`) become a
CASA-style multi-beam `BEAMS` binary-table extension in each beam-carrying file, whose
fixed-size rows the tasks also write at disjoint offsets. The resulting files open in any
FITS viewer (CARTA, DS9) and round-trip through XRADIO's FITS reader
(`xradio.image.open_image`), reconstructing the same coordinates as the Zarr store.

FITS output requires `skunk_works=True` (it is written by the direct-`pwrite`
skunk-works path in `astroviper.node_tasks.imaging.utils.skunk_works_fits`). Boolean
masks are stored as 0.0/1.0 floats (FITS images have no boolean type), and uv-domain /
complex variables have no FITS representation.

In [ ]:
FITS_IMAGE_STORE = "twhya_clean.img.fits"

return_dict_fits = image_cube_single_field(
    ps_store=PS_STORE,
    image_store=FITS_IMAGE_STORE,
    image_params=image_params,
    imaging_weights_params=imaging_weights_params,
    iteration_control_params=clean_iteration_control,
    gridder="prolate_spheroidal",
    deconvolver="hogbom_many_threads",
    scan_intents="OBSERVE_TARGET#ON_SOURCE",
    image_data_variables_keep=[
        "sky_residual",
        "sky_model",
        "mask",
        "point_spread_function",
        "primary_beam",
        "beam_fit_params_point_spread_function",
    ],
    processing_set_data_group_name="base",
    single_precision_image=True,
    processing_function_threads=1,
    n_mapping_parallelism={"frequency": 5},
    overwrite=True,
    restore=True,
    skunk_works=True,  # FITS output uses the direct-pwrite skunk-works path
    output_image_format="fits",
)

print("FITS files written:")
for file_name in sorted(os.listdir(FITS_IMAGE_STORE)):
    size_mb = os.path.getsize(os.path.join(FITS_IMAGE_STORE, file_name)) / 1e6
    print(f"  {file_name:35s} {size_mb:6.1f} MB")

In [ ]:
plot_task_stream(return_dict_fits);

### Reading the FITS output back

`xradio.image.open_image` reads a FITS image into the same image-dataset layout as the
Zarr store (pass a `{image_type: path}` dict to label the variable). Below we open the
restored image, check its header with astropy, verify the pixels match the Zarr run
bit-for-bit, and confirm the per-channel clean beams round-tripped through the `BEAMS`
table.

In [ ]:
from astropy.io import fits
from xradio.image import open_image

restored_fits = os.path.join(FITS_IMAGE_STORE, "SKY_RESTORED.fits")

with fits.open(restored_fits) as hdulist:
    hdulist.info()
    header = hdulist[0].header
    print()
    for key in [
        "CTYPE1",
        "CTYPE2",
        "CTYPE3",
        "CTYPE4",
        "RADESYS",
        "SPECSYS",
        "RESTFRQ",
        "BUNIT",
        "TELESCOP",
        "DATE-OBS",
        "CASAMBM",
    ]:
        print(f"  {key:9s} = {header[key]}")

fits_xds = open_image({"sky": restored_fits})

# The FITS round trip must match the Zarr store exactly (identical pixels).
zarr_restored = clean_xds["SKY_RESTORED"]
fits_restored = fits_xds["SKY"].transpose("time", "frequency", "polarization", "l", "m")
print(
    "\nmax |FITS - Zarr| (SKY_RESTORED):",
    float(np.nanmax(np.abs(fits_restored.values - zarr_restored.values))),
)
print(
    "clean beams round-trip:",
    np.allclose(
        fits_xds["BEAM_FIT_PARAMS_SKY"].values,
        clean_xds["BEAM_FIT_PARAMS_POINT_SPREAD_FUNCTION"].values,
    ),
)

chan = 2
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2), constrained_layout=True)
for ax, (label, plane) in zip(
    axes,
    [
        (
            "SKY_RESTORED (Zarr store)",
            zarr_restored.isel(time=0, frequency=chan, polarization=0),
        ),
        (
            "SKY_RESTORED (FITS via xradio)",
            fits_restored.isel(time=0, frequency=chan, polarization=0),
        ),
    ],
    strict=False,
):
    im = ax.imshow(plane.values, origin="lower", cmap="viridis")
    ax.set_title(f"{label}\nchannel {chan}, Stokes I")
    fig.colorbar(im, ax=ax)
plt.show()

## 8. How the imager is built: the four layers

Everything you just ran is organized into AstroVIPER's four layers, where calls only ever
go **downward**:

```
src/astroviper/
├── distributed_applications/  (1) Builds + computes the Dask graph (GraphVIPER map/reduce)
│                                  and creates the empty on-disk output structures.
│                                  The image_cube_single_field you called lives here.
├── node_tasks/                (2) The functions that run as graph nodes — one per frequency
│                                  chunk here. They load their chunk of visibilities, call
│                                  processing functions, and write their slice of the image.
├── processing_functions/      (3) The stateless science: gridders, weighting, FFT
│                                  normalization, deconvolvers (C++ kernels), restore.
│                                  No I/O, no Dask.
└── utils/                     (4) Cross-cutting helpers: data-group tools, Zarr I/O helpers.
```

The imaging entry point is exposed at all three code layers under the same name,
`image_cube_single_field` — the driver you called, the node task it maps over frequency
chunks, and the processing function that runs the CLEAN loop of residual update and model
update cycles. There are two independent levels of parallelism: **across chunks**
(`n_mapping_parallelism` Dask tasks) and **within a task** (`processing_function_threads`, passed as
`processing_function_threads` to the C++ kernels).

For layer rules, the data-group tooling, the Python ↔ C++ memory contract, and the full
imaging deep-dive, read `AGENTS.md` at the repository root.

In [ ]:
viper_client.close()